# Data-centric AI: better labels, better accuracy

**Question:** Can we improve a model by improving the *training data* while keeping the model exactly the same?

We will classify handwritten digits (0–9) using scikit-learn's built-in `digits` dataset. We deliberately introduce mistakes into **30% of the training labels**, then simulate a human review that corrects them. We compare the same **3-nearest-neighbours (3-NN)** classifier before and after the review on an untouched test set.

**Run in Google Colab:** Upload this `.ipynb` file, then choose **Runtime → Run all**. It uses packages already available in Colab and does not download any data.


## 1. Load the data and make a held-out test set

Each example is an 8 × 8 image of a handwritten digit. We split the data **before** introducing label errors; the test images and labels stay untouched.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

digits = load_digits()
X_train, X_test, y_train_true, y_test = train_test_split(
    digits.data, digits.target,
    test_size=0.25, stratify=digits.target, random_state=42
)

print(f"Training examples: {len(X_train):,}")
print(f"Untouched test examples: {len(X_test):,}")
print(f"Classes: {[int(c) for c in digits.target_names]}")


### Visual check: what does the clean dataset look like?

Here is one training image from each digit class. The small images make label mistakes easy to understand.


In [ ]:
example_idx = [np.flatnonzero(y_train_true == digit)[0] for digit in range(10)]
fig, axes = plt.subplots(2, 5, figsize=(10, 5.5))
for digit, (ax, i) in enumerate(zip(axes.flat, example_idx)):
    ax.imshow(X_train[i].reshape(8, 8), cmap="gray_r", interpolation="nearest")
    ax.set_title(f"Digit {digit}", fontsize=12)
    ax.axis("off")
fig.suptitle("One correctly labelled training image per class", fontsize=14)
fig.subplots_adjust(top=0.86, hspace=0.48)
plt.show()


## 2. Simulate a data-quality problem

A recorded label might say “3” even though the image is a “5”. We randomly change 30% of **training labels** to a different digit. The original labels are retained only so we can simulate a later human audit.


In [ ]:
rng = np.random.default_rng(7)
noise_fraction = 0.30
n_corrupted = int(noise_fraction * len(y_train_true))
corrupted_idx = rng.choice(len(y_train_true), size=n_corrupted, replace=False)

# An offset from 1 to 9 guarantees a different digit label (0–9).
y_train_noisy = y_train_true.copy()
offsets = rng.integers(1, 10, size=n_corrupted)
y_train_noisy[corrupted_idx] = (y_train_noisy[corrupted_idx] + offsets) % 10

print(f"Incorrect training labels: {n_corrupted:,} / {len(y_train_noisy):,} "
      f"({100 * n_corrupted / len(y_train_noisy):.1f}%)")
print("Test labels changed: 0")

fig, axes = plt.subplots(2, 6, figsize=(12, 5))
for col, i in enumerate(corrupted_idx[:6]):
    for row in (0, 1):
        axes[row, col].imshow(X_train[i].reshape(8, 8), cmap="gray_r", interpolation="nearest")
        axes[row, col].axis("off")
    axes[0, col].set_title(f"Noisy: {y_train_noisy[i]}", color="#b73b32", fontsize=11)
    axes[1, col].set_title(f"Reviewed: {y_train_true[i]}", color="#087e79", fontsize=11)
fig.suptitle("Same six images: incorrect recorded label (top), corrected label (bottom)", fontsize=13)
plt.tight_layout()
plt.show()


### Visual check: how widespread are the label errors?

The orange part of each bar counts intentionally corrupted labels for that **actual** digit. We know the correct classes only because this is a controlled simulation.


In [ ]:
total_by_digit = np.bincount(y_train_true, minlength=10)
wrong_by_digit = np.bincount(y_train_true[corrupted_idx], minlength=10)
correct_by_digit = total_by_digit - wrong_by_digit

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(10), correct_by_digit, color="#168b88", label="Correct recorded label")
ax.bar(range(10), wrong_by_digit, bottom=correct_by_digit,
       color="#d97757", label="Incorrect recorded label")
ax.set_xticks(range(10))
ax.set_xlabel("Actual digit")
ax.set_ylabel("Number of training images")
ax.set_title("Training-label quality before review")
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.18))
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
fig.subplots_adjust(bottom=0.25, top=0.88)
plt.show()


## 3. Train the simple model on the noisy data

We use **3-NN** with `n_neighbors=3`. This setting is fixed in advance and will not change when we improve the labels.


In [ ]:
model_before = KNeighborsClassifier(n_neighbors=3)
model_before.fit(X_train, y_train_noisy)
pred_before = model_before.predict(X_test)
accuracy_before = accuracy_score(y_test, pred_before)
print(f"Accuracy with noisy training labels: {accuracy_before:.1%}")


## 4. Improve the training data, then retrain the same model

Imagine a human checks every training image and corrects the recorded digit. In this **controlled simulation**, the dataset's original labels stand in for that human review. A real project would need actual reviewers or another trusted source; the model does not secretly know the correct labels.

Only the training labels change. We use **the same images, test set, algorithm and `n_neighbors=3`**.


In [ ]:
y_train_audited = y_train_true.copy()  # Simulated human review of training labels only

model_after = KNeighborsClassifier(n_neighbors=3)
model_after.fit(X_train, y_train_audited)
pred_after = model_after.predict(X_test)
accuracy_after = accuracy_score(y_test, pred_after)

gain_pp = 100 * (accuracy_after - accuracy_before)
print(f"Before label review: {accuracy_before:.1%}")
print(f"After label review:  {accuracy_after:.1%}")
print(f"Improvement:         +{gain_pp:.1f} percentage points")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
values = [100 * accuracy_before, 100 * accuracy_after]
bars = ax.bar(["Noisy training labels", "Audited training labels"],
              values, color=["#d97757", "#168b88"], width=0.55)
ax.set_ylim(0, 105)
ax.set_ylabel("Accuracy on untouched test set (%)")
ax.set_title("Same 3-NN model, better training labels")
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in values], padding=4, fontsize=12)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()


### Visual check: which test predictions improved?

These are held-out test images that the model with noisy labels got wrong but the **same model**, retrained with reviewed labels, got right. We only inspect them **after** both models have been evaluated; they were never used in the training-label review.


In [ ]:
improved_idx = np.flatnonzero((pred_before != y_test) & (pred_after == y_test))
fig, axes = plt.subplots(2, 4, figsize=(12, 7.5))
for ax, i in zip(axes.flat, improved_idx[:8]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r", interpolation="nearest")
    ax.set_title(f"Actual: {y_test[i]}\nBefore: {pred_before[i]}  →  After: {pred_after[i]}", fontsize=10)
    ax.axis("off")
fig.suptitle("Examples of corrected predictions on untouched test images", fontsize=14)
fig.subplots_adjust(left=0.04, right=0.98, top=0.88, bottom=0.05,
                    hspace=0.62, wspace=0.25)
plt.show()
print(f"Test examples changed from wrong to correct: {len(improved_idx)}")


## What does this show?

- **Data-centric change:** Correct the training labels. We did not make the model more complex or change its settings.
- **Fair comparison:** Both versions are measured on the **same untouched test set**. Its labels were never used to decide which training examples to correct.
- **Teaching caveat:** We injected unusually many label errors so the effect is easy to see. A real dataset may have fewer errors, and a real audit takes human effort.

**Question:** If you could only review 100 training examples, how would you decide which labels to check first?
